In [31]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import utils

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Application Train/Test

In [32]:
application = pd.read_csv('../data/raw/application_train.csv', sep=',')

In [33]:
# On bascule a nan pour l'outlier days = 365243
application['DAYS_EMPLOYED'] = application['DAYS_EMPLOYED'].replace(365243, np.nan)

gender_map = {'M': 0, 'F': 1}
own_car_map = {'N': 0, 'Y': 1}
own_realty_map = {'N': 0, 'Y': 1}
education_type_map = {
    'Lower secondary': 0,
    'Secondary / secondary special': 1,
    'Incomplete higher': 2,
    'Higher education': 3,
    'Academic degree': 4
}

application['code_gender'] = application['CODE_GENDER'].map(gender_map)
application['own_car'] = application['FLAG_OWN_CAR'].map(own_car_map)
application['own_realty'] = application['FLAG_OWN_REALTY'].map(own_realty_map)
application['education_type'] = application['NAME_EDUCATION_TYPE'].map(education_type_map)

application['ratio_days_employed_vs_days_birth'] = application['DAYS_EMPLOYED'] / application['DAYS_BIRTH']
application['ratio_income_vs_credit'] = application['AMT_INCOME_TOTAL'] / application['AMT_CREDIT']
application['ratio_income_total_vs_fam_number'] = application['AMT_INCOME_TOTAL'] / application['CNT_FAM_MEMBERS']
application['ratio_annuity_vs_income_total'] = application['AMT_ANNUITY'] / application['AMT_INCOME_TOTAL']
application['ratio_children_vs_family'] = application['CNT_CHILDREN'] / application['CNT_FAM_MEMBERS']
application['age'] = application['DAYS_BIRTH'] / -365.25

application = pd.get_dummies(application, [col for col in application.columns if application[col].dtype == 'object'])

In [34]:
application = utils.add_prefix_to_cols('app_', application)
utils.save_data_file('./../data/interim', 'application.csv', application)

## Bureau Balance

In [35]:
bureau_balance = pd.read_csv('../data/raw/bureau_balance.csv', sep=',')

In [36]:
# Aggregation de bureau_balance
status_map = {'C': 0, 'X': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bureau_balance['STATUS_NUM'] = bureau_balance['STATUS'].map(status_map)

# Agrégation par prêt
bureau_balance_aggr = bureau_balance.groupby('SK_ID_BUREAU').agg(
    max_status=('STATUS_NUM', 'max'),
    nb_months_with_delay=('STATUS_NUM', lambda x: (x > 0).sum())
).reset_index()

In [37]:
bureau_balance_aggr = utils.add_prefix_to_cols('bur_bal_', bureau_balance_aggr)
utils.save_data_file('./../data/interim', 'bureau_balance_aggr.csv', bureau_balance_aggr)

## Bureau

In [38]:
bureau = pd.read_csv('../data/raw/bureau.csv', sep=',')

In [39]:
bureau = pd.get_dummies(bureau, columns=['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE'])

bureau['AMT_CREDIT_ACTIVE'] = bureau['AMT_CREDIT_SUM'] * bureau['CREDIT_ACTIVE_Active']
bureau['AMT_CREDIT_CLOSED'] = bureau['AMT_CREDIT_SUM'] * bureau['CREDIT_ACTIVE_Closed']

type_cols = [c for c in bureau.columns if c.startswith('CREDIT_TYPE_')]
agg_type = {f'type_{col.lower().replace(' ', '_').replace('(', '').replace(')', '')}': (col, 'sum') for col in type_cols}

currency_cols = [c for c in bureau.columns if c.startswith('CREDIT_CURRENCY_')]
agg_currency = {f'{col.lower().replace('credit_currency_', '').replace(' ', '_')}_sum' : (col, 'sum') for col in currency_cols}

bureau_aggr = bureau.groupby('SK_ID_CURR').agg(
    SK_ID_BUREAU =  ('SK_ID_BUREAU', 'sum'),
    credit_active_closed = ('CREDIT_ACTIVE_Closed', 'sum'),
    credit_active_active = ('CREDIT_ACTIVE_Active', 'sum'),
    credit_active_bad_debt = ('CREDIT_ACTIVE_Bad debt', 'sum'),
    amt_credit_sum = ('AMT_CREDIT_SUM', 'sum'),
    amt_credit_sum_debt = ('AMT_CREDIT_SUM_DEBT', 'sum'),
    amt_credit_sum_overdue = ('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    credit_day_overdue = ('CREDIT_DAY_OVERDUE', 'sum'),
    days_credit_min = ('DAYS_CREDIT', 'min'),
    days_credit_max = ('DAYS_CREDIT', 'max'),
    days_credit_mean = ('DAYS_CREDIT', 'mean'),
    amt_credit_active_active_sum = ('AMT_CREDIT_ACTIVE', 'sum'),
    amt_credit_active_closed_sum = ('AMT_CREDIT_CLOSED', 'sum'),

    **agg_currency,
    **agg_type

).reset_index()


In [40]:
bureau_aggr = utils.add_prefix_to_cols('bur_', bureau_aggr)
utils.save_data_file('./../data/interim', 'bureau_aggr.csv', bureau_aggr)

## Installments Payments

In [41]:
installments_payments = pd.read_csv('../data/raw/installments_payments.csv', sep=',')

In [42]:
month_threshold = -30

condition_payment_default = ((installments_payments['AMT_PAYMENT'].isnull()) & (installments_payments['DAYS_INSTALMENT'] < month_threshold))
condition_payment_pending = ((installments_payments['AMT_PAYMENT'].isnull()) & (installments_payments['DAYS_INSTALMENT'] >= month_threshold))

installments_payments['is_payment_default'] = condition_payment_default.astype(int)
installments_payments['is_payment_pending'] = condition_payment_pending.astype(int)
installments_payments['delta_payment_days'] = installments_payments['DAYS_INSTALMENT'] - installments_payments['DAYS_ENTRY_PAYMENT']

installments_payments.loc[condition_payment_pending, 'AMT_PAYMENT'] = np.nan
installments_payments.loc[condition_payment_pending, 'delta_payment_days'] = np.nan

installments_payments_aggr = installments_payments.groupby(['SK_ID_CURR', 'SK_ID_PREV', 'NUM_INSTALMENT_NUMBER']).agg(
    payment_count=('SK_ID_PREV', 'count'),
    is_payment_default = ('is_payment_default', 'max'),
    is_payment_pending = ('is_payment_pending', 'max'),
    amt_payment_sum=('AMT_PAYMENT', 'sum'),
    amt_instalment_mean=('AMT_INSTALMENT', 'mean'),
    delta_payment_days_mean=('delta_payment_days', 'mean'),
).reset_index()

installments_payments_aggr['is_fractional'] = (installments_payments_aggr['payment_count'] > 1).astype(int)

installments_payments_aggr = installments_payments_aggr.groupby('SK_ID_CURR').agg(
    payment_count=('payment_count', 'max'),
    amt_payment_sum=('amt_payment_sum', 'max'),
    amt_instalment_mean=('amt_instalment_mean', 'max'),
    delta_payment_days_mean=('delta_payment_days_mean', 'max'),

    total_fractional_payments=('is_fractional', 'sum'),
    total_instalments=('is_fractional', 'count')
)

installments_payments_aggr['fractional_payment_ratio'] = (installments_payments_aggr['total_fractional_payments'] / installments_payments_aggr['total_instalments'])

installments_payments_aggr = installments_payments_aggr.reset_index()

In [43]:
installments_payments_aggr = utils.add_prefix_to_cols('ins_pay_', installments_payments_aggr)
utils.save_data_file('./../data/interim', 'installments_payments_aggr.csv', installments_payments_aggr)

## Pos Cash Balance

In [44]:
pos_cash_balance = pd.read_csv('../data/raw/POS_CASH_balance.csv', sep=',')

In [45]:
pos_cash_balance['NAME_CONTRACT_STATUS'].value_counts()
good_contract_name_status = ['Completed']
bad_contract_name_status = ['Demand', 'Returned to the store', 'Amortized debt']

pos_cash_balance['month_with_dpd_def'] = (pos_cash_balance['SK_DPD_DEF'] > 0).astype(int)
pos_cash_balance['good_contract_name_status'] = (pos_cash_balance['NAME_CONTRACT_STATUS']).isin(good_contract_name_status).astype(int)
pos_cash_balance['bad_contract_name_status'] = (pos_cash_balance['NAME_CONTRACT_STATUS']).isin(bad_contract_name_status).astype(int)

pos_cash_balance_aggr = pos_cash_balance.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
    days_past_due_sum = ('SK_DPD', 'sum'),
    days_past_due_def_sum = ('SK_DPD_DEF', 'sum'),
    month_with_dpd_def_sum = ('month_with_dpd_def', 'sum'),
    good_contract_name_status = ('good_contract_name_status', 'sum'),
    bad_contract_name_status = ('bad_contract_name_status', 'sum'),
    cnt_instalment_min = ('CNT_INSTALMENT', 'min'),
    cnt_instalment_max = ('CNT_INSTALMENT', 'max')
).reset_index()

pos_cash_balance_aggr['delta_cnt_instalment'] = pos_cash_balance_aggr['cnt_instalment_max'] - pos_cash_balance_aggr['cnt_instalment_min']

pos_cash_balance_aggr = pos_cash_balance_aggr.groupby('SK_ID_CURR').agg(
    total_days_past_due = ('days_past_due_sum', 'sum'),
    total_days_past_due_def = ('days_past_due_def_sum', 'sum'),
    total_months_with_dpd = ('month_with_dpd_def_sum', 'sum'),
    total_good_status = ('good_contract_name_status', 'sum'),
    total_bad_status = ('bad_contract_name_status', 'sum'),
    max_term_extension = ('delta_cnt_instalment', 'max')
).reset_index()

In [46]:
pos_cash_balance_aggr = utils.add_prefix_to_cols('pos_cash_bal_', pos_cash_balance_aggr)
utils.save_data_file('./../data/interim', 'pos_cash_balance_aggr.csv', pos_cash_balance_aggr)

## Credit Card balance

In [47]:
credit_card_balance = pd.read_csv('../data/raw/credit_card_balance.csv', sep=',')

In [48]:
credit_card_balance = pd.get_dummies(credit_card_balance, columns=['NAME_CONTRACT_STATUS'])

name_contract_status_cols = [c for c in credit_card_balance.columns if c.startswith('NAME_CONTRACT_STATUS_')]
agg_name_contract_status_cols = {f'type_{col.lower().replace(' ', '_')}': (col, 'sum') for col in name_contract_status_cols}

credit_card_balance['credit_card_utilisation_ratio'] = (
    credit_card_balance['AMT_BALANCE'] / (credit_card_balance['AMT_CREDIT_LIMIT_ACTUAL'] + 0.001)
)

credit_card_balance['credit_card_minimum_payment_ratio'] = (
    credit_card_balance['AMT_PAYMENT_CURRENT'] / (credit_card_balance['AMT_INST_MIN_REGULARITY'] + 0.001)
)

credit_card_balance['credit_card_atm_ratio'] = (
    credit_card_balance['AMT_DRAWINGS_ATM_CURRENT'] / (credit_card_balance['AMT_DRAWINGS_CURRENT'] + 0.001)
)

credit_card_balance_aggr = credit_card_balance.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
    amt_balance_mean = ('AMT_BALANCE', 'mean'),
    amt_credit_limit_mean = ('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
    sk_dpd_sum = ('SK_DPD', 'sum'),
    sk_dpd_def_sum = ('SK_DPD_DEF', 'sum'),
    sk_dpd_max = ('SK_DPD', 'max'),
    sk_dpd_def_max = ('SK_DPD_DEF', 'max'),
    cnt_drawings_current_mean = ('CNT_DRAWINGS_CURRENT', 'mean'),
    cnt_drawings_atm_current_mean = ('CNT_DRAWINGS_ATM_CURRENT', 'mean'),

    credit_card_utilisation_ratio = ('credit_card_utilisation_ratio', 'mean'),
    credit_card_atm_ratio = ('credit_card_atm_ratio', 'mean'),
    credit_card_minimum_payment_ratio = ('credit_card_minimum_payment_ratio', 'mean'),

    **agg_name_contract_status_cols,

).reset_index()

new_status_cols = list(agg_name_contract_status_cols.keys())
agg_name_contract_status_cols_2 = {col: (col, 'sum') for col in new_status_cols}

credit_card_balance_aggr = credit_card_balance_aggr.groupby('SK_ID_CURR').agg(
    total_balance_mean = ('amt_balance_mean', 'sum'),
    total_credit_limit_mean = ('amt_credit_limit_mean', 'sum'),
    sk_dpd_sum = ('sk_dpd_sum', 'sum'),
    sk_dpd_def_sum = ('sk_dpd_def_sum', 'sum'),
    sk_dpd_max = ('sk_dpd_max', 'max'),
    sk_dpd_def_max = ('sk_dpd_def_max', 'max'),
    cnt_drawings_current_mean = ('cnt_drawings_current_mean', 'mean'),
    cnt_drawings_atm_current_mean = ('cnt_drawings_atm_current_mean', 'mean'),

    credit_card_utilisation_ratio = ('credit_card_utilisation_ratio', 'mean'),
    credit_card_atm_ratio = ('credit_card_atm_ratio', 'mean'),
    credit_card_minimum_payment_ratio = ('credit_card_minimum_payment_ratio', 'mean'),

    **agg_name_contract_status_cols_2,

).reset_index()

In [49]:
credit_card_balance_aggr = utils.add_prefix_to_cols('cred_card_bal_', credit_card_balance_aggr)
utils.save_data_file('./../data/interim', 'credit_card_balance_aggr.csv', credit_card_balance_aggr)

## Previous Application

In [50]:
previous_application = pd.read_csv('../data/raw/previous_application.csv', sep=',')

In [51]:
previous_application['DAYS_FIRST_DRAWING'] = previous_application['DAYS_FIRST_DRAWING'].replace(365243, np.nan)
previous_application['DAYS_FIRST_DUE'] = previous_application['DAYS_FIRST_DUE'].replace(365243, np.nan)
previous_application['DAYS_LAST_DUE_1ST_VERSION'] = previous_application['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan)
previous_application['DAYS_LAST_DUE'] = previous_application['DAYS_LAST_DUE'].replace(365243, np.nan)
previous_application['DAYS_TERMINATION'] = previous_application['DAYS_TERMINATION'].replace(365243, np.nan)

previous_application['delta_demanded_accepted'] = previous_application['AMT_APPLICATION'] - previous_application['AMT_CREDIT']
previous_application['ratio_down_payment'] = previous_application['AMT_APPLICATION'] - previous_application['AMT_DOWN_PAYMENT']

previous_application = pd.get_dummies(previous_application, columns=['NAME_CONTRACT_STATUS', 'CODE_REJECT_REASON', 'WEEKDAY_APPR_PROCESS_START'])

name_contract_status_cols = [c for c in previous_application.columns if c.startswith('NAME_CONTRACT_STATUS_')]
agg_name_contract_status_cols = {f'{col.lower().replace(' ', '_')}': (col, 'sum') for col in name_contract_status_cols}

reject_reason_code_cols = [c for c in previous_application.columns if c.startswith('CODE_REJECT_REASON_')]
agg_reject_reason_code_cols = {f'{col.lower().replace(' ', '_')}': (col, 'sum') for col in reject_reason_code_cols}

weekdays_cols = [c for c in previous_application.columns if c.startswith('WEEKDAY_APPR_PROCESS_START_')]
agg_weekdays_cols = {f'{col.lower().replace(' ', '_')}': (col, 'sum') for col in weekdays_cols}

previous_application_aggr = previous_application.groupby('SK_ID_CURR').agg(
    application_count = ('SK_ID_PREV', 'count'),
    delta_demanded_accepted_sum = ('delta_demanded_accepted', 'sum'),
    ratio_down_payment_mean = ('ratio_down_payment', 'mean'),
    days_decision_min = ('DAYS_DECISION', 'min'),
    nflag_insured_on_approval_mean = ('NFLAG_INSURED_ON_APPROVAL', 'mean'),
    amt_annuity_mean = ('AMT_ANNUITY', 'mean'),
    amt_annuity_min = ('AMT_ANNUITY', 'min'),
    amt_annuity_max = ('AMT_ANNUITY', 'max'),
    hour_process_mean = ('HOUR_APPR_PROCESS_START', 'mean'),
    hour_process_min = ('HOUR_APPR_PROCESS_START', 'min'),
    hour_process_max = ('HOUR_APPR_PROCESS_START', 'max'),
    rate_down_payment_mean = ('RATE_DOWN_PAYMENT', 'mean'),
    rate_down_payment_min = ('RATE_DOWN_PAYMENT', 'min'),
    rate_down_payment_max = ('RATE_DOWN_PAYMENT', 'max'),
    amt_goods_price_mean = ('AMT_GOODS_PRICE', 'mean'),
    amt_goods_price_min = ('AMT_GOODS_PRICE', 'min'),
    amt_goods_price_max = ('AMT_GOODS_PRICE', 'max'),
    amt_application_mean = ('AMT_APPLICATION', 'mean'),
    amt_application_min = ('AMT_APPLICATION', 'min'),
    amt_application_max = ('AMT_APPLICATION', 'max'),
    amt_credit_mean = ('AMT_CREDIT', 'mean'),
    amt_credit_min = ('AMT_CREDIT', 'min'),
    amt_credit_max = ('AMT_CREDIT', 'max'),
    cnt_payment_mean = ('CNT_PAYMENT', 'mean'),
    cnt_payment_sum = ('CNT_PAYMENT', 'sum'),

    **agg_name_contract_status_cols,
    **agg_reject_reason_code_cols,
    **agg_weekdays_cols

).reset_index()

previous_application_aggr['ratio_refused_vs_total'] = previous_application_aggr['name_contract_status_refused'] / previous_application_aggr['application_count']
previous_application_aggr['ratio_approved_vs_total'] = previous_application_aggr['name_contract_status_approved'] / previous_application_aggr['application_count']


In [52]:
previous_application_aggr = utils.add_prefix_to_cols('prev_app_', previous_application_aggr)
utils.save_data_file('./../data/interim', 'previous_application_aggr.csv', previous_application_aggr)

In [53]:
application = pd.read_csv('../data/interim/application.csv', sep=',')
bureau_balance = pd.read_csv('../data/interim/bureau_balance_aggr.csv', sep=',')
bureau = pd.read_csv('../data/interim/bureau_aggr.csv', sep=',')
credit_card_balance = pd.read_csv('../data/interim/credit_card_balance_aggr.csv', sep=',')
installments_payment = pd.read_csv('../data/interim/installments_payments_aggr.csv', sep=',')
pos_cash_balance = pd.read_csv('../data/interim/pos_cash_balance_aggr.csv', sep=',')
previous_applications = pd.read_csv('../data/interim/previous_application_aggr.csv', sep=',')

In [54]:
bureau = bureau.merge(bureau_balance, how='left', on='SK_ID_BUREAU')
application = application.merge(bureau, how='left', on='SK_ID_CURR')
application = application.merge(credit_card_balance, how='left', on='SK_ID_CURR')
application = application.merge(installments_payment, how='left', on='SK_ID_CURR')
application = application.merge(previous_applications, how='left', on='SK_ID_CURR')
application = application.merge(pos_cash_balance, how='left', on='SK_ID_CURR')

In [55]:
utils.save_data_file('../data/rafined', 'application.csv', application)

In [56]:
cols = [col for col in application.columns if application[col].dtype == 'object']
print(f'Colonnes catégorielles restantes : {len(cols)}')
print(cols)

Colonnes catégorielles restantes : 0
[]
